In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import argparse
from collections import defaultdict
import math
import sys

In [ ]:
import csv

input_file = "Haitian_Name_Adjectives_Story1.csv"
output_file = "Haitian_Name_Adjectives_Story.csv"

with open(input_file, mode="r", encoding="utf-8") as infile, \
     open(output_file, mode="w", newline="", encoding="utf-8") as outfile:

    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    for row in reader:
        cleaned_row = [cell.replace("\n", " ") if cell else cell for cell in row]
        writer.writerow(cleaned_row)

In [ ]:
import csv
import re

input_file = "Haitian Artists Extract Story.csv"
output_file = "Haitian_Artists_adjective_Story.csv"

resultats = []

with open(input_file, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)

    for row in reader:
        texte = row[0]

        match = re.search(r'1\.(.*?)2\.', texte, re.DOTALL)
        if match:
            contenu = match.group(1).strip()
            resultats.append([contenu])
with open(output_file, "w", newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(resultats)

In [ ]:
df1 = pd.read_csv('Haitian_Foods_adjective_Story.csv', header=None)
df2 = pd.read_csv('French_Foods_adjective_Story.csv',header=None)

In [ ]:
df1 = df1[:191]

In [ ]:
df0 = pd.concat([df1, df2], ignore_index=True)

In [ ]:
df1

,0
0,"Solèy la te kòmanse kouri sou tè a, yon limyè ..."
1,"Solèy la te kòmanse monte sou tè a, yon lanmou..."
2,"Solèy la te kòmanse kouche sou tè a, yon koulè..."
3,"Solèy la te kòmanse monte sou tèt lakay la, yo..."
4,"Solèy la te kòmanse kouche sou tè a, yon koulè..."
...,...
186,"Solèy la te kòmanse kouri sou lannwit la, yon ..."
187,"Solèy la te kòmanse leve sou mòn yo, yon limyè..."
188,"Solèy la te kòmanse monte sou lèt la, yon koul..."
189,"Solèy la te kòmanse kouri sou tèt kay la, yon ..."


In [ ]:
def get_text_column(df):
    if isinstance(df, pd.Series):
        return df
    else:
        return df.iloc[:, 0]

In [ ]:
df1 = get_text_column(df1)
df2 = get_text_column(df2)
df0 = get_text_column(df0)

In [ ]:
STOPWORDS_HT = set([
    "mwen", "ou", "li", "nou", "yo",
    "sa", "ki", "kisa",
    "nan", "sou", "ak", "pou", "pa",
    "se", "ye", "te", "ap", "pral",
    "gen", "fè", "di",
    "la", "a", "an", "lan",
    "yon", "youn",
    "men", "oswa", "paske",
    "kòm", "tankou",
    "isit", "la", "là",
    "tout", "plis", "mwens",
    "byen", "mal", "m", "ka","l", " ","t","w",
    "vin","vini","k","san","manje","moun", "si","ale","tt","sak","pi",""
])

In [ ]:
def get_log_odds(col1, col2, col0,verbose=False,lower=True):
    """Monroe et al. Fightin' Words method to identify top words in df1 and df2
    against df0 as the background corpus"""
    if lower:
        counts1 = defaultdict(int, [[i,j] for i,j in col1.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        counts2 = defaultdict(int,[[i,j] for i,j in col2.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        prior = defaultdict(int,[[i,j] for i,j in col0.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
    else:
        counts1 = defaultdict(int,[[i,j] for i,j in col1.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        counts2 = defaultdict(int,[[i,j] for i,j in col2.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        prior = defaultdict(int,[[i,j] for i,j in col0.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])

    sigmasquared = defaultdict(float)
    sigma = defaultdict(float)
    delta = defaultdict(float)

    for word in prior.keys():
        prior[word] = int(prior[word] + 0.5)

    for word in counts2.keys():
        counts1[word] = int(counts1[word] + 0.5)
        if prior[word] == 0:
            prior[word] = 1

    for word in counts1.keys():
        counts2[word] = int(counts2[word] + 0.5)
        if prior[word] == 0:
            prior[word] = 1

    n1 = sum(counts1.values())
    n2 = sum(counts2.values())
    nprior = sum(prior.values())

    for word in prior.keys():
        if prior[word] > 0:
            l1 = float(counts1[word] + prior[word]) / (( n1 + nprior ) - (counts1[word] + prior[word]))
            l2 = float(counts2[word] + prior[word]) / (( n2 + nprior ) - (counts2[word] + prior[word]))
            sigmasquared[word] =  1/(float(counts1[word]) + float(prior[word])) + 1/(float(counts2[word]) + float(prior[word]))
            sigma[word] =  math.sqrt(sigmasquared[word])
            delta[word] = ( math.log(l1) - math.log(l2) ) / sigma[word]

    if verbose:
        for word in sorted(delta, key=delta.get)[:10]:
            print("%s, %.3f" % (word, delta[word]))

        for word in sorted(delta, key=delta.get,reverse=True)[:10]:
            print("%s, %.3f" % (word, delta[word]))
    return delta

In [ ]:
food_results = get_log_odds(get_text_column(df1),get_text_column(df2),get_text_column(df0),verbose=False,lower=True)

In [ ]:
haitian_food_dict = defaultdict(float)
french_food_dict = defaultdict(float)
for word , score  in food_results.items():
  if score > 1 and word not in STOPWORDS_HT:
    haitian_food_dict[word] = food_results[word]
  if score <- 1  and word not in STOPWORDS_HT:
    french_food_dict[word] = food_results[word]

In [ ]:
haitian_food_dict= dict(sorted(haitian_food_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_food_dict

{'rose': 3.3535182486117674,
 'manman': 2.629928685365599,
 'mama': 1.952459896605969,
 'kb': 1.935215538855418,
 'mayi': 1.8692802858446769,
 'apre': 1.698887447101065,
 'gwo': 1.5668070501601525,
 'pwa': 1.5180424266294028,
 'relasyon': 1.507660004978913,
 'bol': 1.4909855807515102,
 'kay': 1.4605002137436638,
 'jis': 1.434828007759117,
 'rann': 1.419124096139424,
 'jean': 1.4117179184259727,
 'envante': 1.3134691205463596,
 'granmoun': 1.3035562484164491,
 'ananas': 1.2600349701506648,
 'bannann': 1.2600349701506648,
 'mabi': 1.2600349701506648,
 'pen': 1.2600349701506648,
 'teren': 1.2056263290570892,
 'jus': 1.2008768347874406,
 'kouraj': 1.126974732429393,
 'bokal': 1.126974732429393,
 'kike': 1.126974732429393,
 'kazo': 1.126974732429393,
 'pouse': 1.126974732429393,
 'taso': 1.126974732429393,
 'pwosesis': 1.126974732429393,
 'diri': 1.126974732429393,
 'elias': 1.126974732429393,
 'kst': 1.126974732429393,
 'konn': 1.0799898074834535,
 'ss': 1.0799898074834535,
 'p': 1.0737751

In [ ]:
haitian_food_dict.keys()

dict_keys(['rose', 'manman', 'mama', 'kb', 'mayi', 'apre', 'gwo', 'pwa', 'relasyon', 'bol', 'kay', 'jis', 'rann', 'jean', 'envante', 'granmoun', 'ananas', 'bannann', 'mabi', 'pen', 'teren', 'jus', 'kouraj', 'bokal', 'kike', 'kazo', 'pouse', 'taso', 'pwosesis', 'diri', 'elias', 'kst', 'konn', 'ss', 'p', 'plan', 'baze', 'dy', 'figi', 'konesans'])

In [ ]:
french_food_dict = dict(sorted(french_food_dict.items(), key=lambda item: item[1]))

In [ ]:
french_food_dict

{'jeanpierre': -2.8049502939349344,
 'elodie': -2.774965741533987,
 'kafe': -2.106698453931023,
 'vilaj': -1.9848405676760834,
 'de': -1.8899477987737472,
 'antoine': -1.8860984538368109,
 'moman': -1.8605989377651941,
 'madame': -1.7982687451411974,
 'dubois': -1.7059348087649435,
 'santt': -1.653840464318849,
 'lanmou': -1.635237833655097,
 'deside': -1.5555543414818815,
 'od': -1.5044001137944567,
 'oranj': -1.5044001137944567,
 'dous': -1.4829052742730047,
 'vant': -1.3414841122627295,
 'krep': -1.2713715915477233,
 'legim': -1.2713715915477233,
 'monten': -1.2191981251086061,
 'pwomt': -1.1604649471993256,
 'kt': -1.1604649471993256,
 'gteau': -1.137114189199419,
 'deplase': -1.137114189199419,
 'koumanse': -1.137114189199419,
 'boulanjeri': -1.137114189199419,
 'elegant': -1.137114189199419,
 'repa': -1.137114189199419,
 'espwa': -1.137114189199419,
 'van': -1.124769522126128,
 'gou': -1.1036035085578786,
 'kl': -1.0965397881634227,
 'bri': -1.0965397881634227,
 'fent': -1.093957

In [ ]:
french_food_dict.keys()

dict_keys(['jeanpierre', 'elodie', 'kafe', 'vilaj', 'de', 'antoine', 'moman', 'madame', 'dubois', 'santt', 'lanmou', 'deside', 'od', 'oranj', 'dous', 'vant', 'krep', 'legim', 'monten', 'pwomt', 'kt', 'gteau', 'deplase', 'koumanse', 'boulanjeri', 'elegant', 'repa', 'espwa', 'van', 'gou', 'kl', 'bri', 'fent', 'vanilye', 'lap', 'antre', 'moso', 'bagay', 'fnwa', 'jodi', 'pasyon'])

Verb Food

In [ ]:
df_food_verb_haitian = pd.read_csv('Haitian_Foods_Verb_Story.csv', header=None)
df_food_verb_french = pd.read_csv('French_Foods_Verb_Story.csv',header=None)
df_food_verb_all = pd.concat([df_food_verb_haitian, df_food_verb_french], ignore_index=True)

In [ ]:
food_verb_results = get_log_odds(get_text_column(df_food_verb_haitian),get_text_column(df_food_verb_french),get_text_column(df_food_verb_all),verbose=False,lower=True)

In [ ]:
haitian_food_verb = {}
french_food_verb = {}
for word , score  in food_verb_results.items():
  if score > 1.75 and word not in STOPWORDS_HT:
    haitian_food_verb[word] = food_verb_results[word]
  if score <- 1.75 and word not in STOPWORDS_HT:
    french_food_verb[word] = food_verb_results[word]

In [ ]:
haitian_food_verb = dict(sorted(haitian_food_verb.items(), key=lambda item: item[1] , reverse=True))
haitian_food_verb

{'diri': 2.933274053740255,
 'pyebwa': 2.904714264092634,
 'bwa': 2.8069971552840847,
 'ss': 2.756392954375194,
 'kiy': 2.55871245997385,
 'mache': 2.4576185383247617,
 'lavi': 2.4232549577563973,
 'chimen': 2.3922788936643533,
 'pwason': 2.2861503770327816,
 'bwi': 2.2766266216298603,
 'marmit': 2.258442211351447,
 'wch': 2.2137164140967758,
 'lari': 2.1985604309821403,
 'granm': 2.193516517832144,
 'lakou': 2.126734043531148,
 'dlo': 2.051687584321382,
 'bouyi': 2.0171268550735992,
 'kasav': 1.9795928602358621,
 'rad': 1.9746489331228512,
 'plastik': 1.8976681817593537,
 'soufle': 1.7925882159876594,
 'kay': 1.786452852979179,
 'tabouret': 1.773451833536953,
 'pate': 1.757110352135371}

In [ ]:
french_food_verb = dict(sorted(french_food_verb.items(), key=lambda item: item[1]))
french_food_verb

{'kote': -4.3283389967273385,
 'psyon': -3.3997033563807064,
 'konn': -3.2275197622094853,
 'fmen': -3.1901936202572267,
 'pote': -3.0670181290065037,
 'rankontre': -2.706085784339563,
 'koupe': -2.7019800765386286,
 'sonje': -2.6315066379840646,
 'ane': -2.0477742226845916,
 'fromaj': -2.0224617331168666,
 'ekri': -1.8612423260658622,
 'ufs': -1.8602667791742156,
 'restoran': -1.8515537079091071,
 'renmen': -1.8056596972416337,
 'canel': -1.7736340355378692,
 'parfm': -1.7674002981328454}

Artist

In [ ]:
df_haitian = pd.read_csv('Haitian_Artists_adjective_Story.csv', header=None)
df_french = pd.read_csv('French_Artists_adjective_Story.csv',header=None)


In [ ]:
df_haitian = df_haitian[:200]

In [ ]:
df_all = pd.concat([df_haitian, df_french], ignore_index=True)

In [ ]:
df_haitian

,0
0,"Solèy la te kòmanse monte sou tèt lakay la, yo..."
1,"Solèy la te kòmanse kouche sou lakay la, yon k..."
2,"Solèy la te kòmanse monte sou lari yo, yon kle..."
3,"Solèy la te kòmanse kouri sou lari yo, yon ti ..."
4,"Solèy la te kòmanse kouri sou montan yo, yon t..."
...,...
195,Solèy la te kòmanse kouri sou vitraj yo nan ti...
196,"Solèy la te kòmanse monte sou tankou sa a, yon..."
197,"Solèy la te kòmanse kouche sou lari yo, yon ko..."
198,"Solèy la te kòmanse kouri sou montan yo, yon l..."


In [ ]:
artist_results = get_log_odds(get_text_column(df_haitian),get_text_column(df_french),get_text_column(df_all),verbose=False,lower=True)

In [ ]:
haitian_artist_dict = {}
french_artist_dict = {}
for word , score  in artist_results.items():
  if score > 1 and word not in STOPWORDS_HT:
    haitian_artist_dict[word] = artist_results[word]
  if score <- 1 and word not in STOPWORDS_HT:
    french_artist_dict[word] = artist_results[word]

In [ ]:
haitian_artist_dict = dict(sorted(haitian_artist_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_artist_dict

{'jean': 2.8556570594885224,
 'rose': 1.9648240099898835,
 'ke': 1.701435934645575,
 'mizik': 1.4246000348818209,
 'chak': 1.3896376968460762,
 'mama': 1.3890928882345257,
 'ritm': 1.3542966972560821,
 'j': 1.268024868859856,
 'anba': 1.2289349812104629,
 'apary': 1.21484819350356,
 'cayes': 1.134122251389217,
 'zv': 1.134122251389217,
 'pasyon': 1.134122251389217,
 'manno': 1.134122251389217,
 'bois': 1.134122251389217,
 'pierre': 1.134122251389217,
 'part': 1.1191320776752733,
 'chz': 1.0952376750285915,
 'rad': 1.0917321348192173,
 'non': 1.0917321348192173,
 'zb': 1.0835146679446115}

In [ ]:
haitian_artist_dict.keys()

dict_keys(['jean', 'rose', 'ke', 'mizik', 'chak', 'mama', 'ritm', 'j', 'anba', 'apary', 'cayes', 'zv', 'pasyon', 'manno', 'bois', 'pierre', 'part', 'chz', 'rad', 'non', 'zb'])

In [ ]:
french_artist_dict = dict(sorted(french_artist_dict.items(), key=lambda item: item[1]))

In [ ]:
french_artist_dict

{'anpil': -2.018707072199149,
 'antoine': -1.786935190698818,
 'elodie': -1.7534131448429382,
 'ekipman': -1.494924097177803,
 'kk': -1.494924097177803,
 'vwa': -1.4246593299727393,
 'depi': -1.3490345902371415,
 'aptman': -1.3286346712295118,
 'b': -1.2633664530293203,
 'jeanpierre': -1.2562857219513026,
 'tann': -1.2232741400026,
 'playlist': -1.1829898892476753,
 'kase': -1.1802511765361454,
 'lonbraj': -1.1299557662065969,
 'd': -1.1299557662065969,
 'tab': -1.0805918366494436,
 'anj': -1.0778361799419576,
 'melanje': -1.0778361799419576,
 'lanp': -1.0758780972731177,
 'estidyo': -1.0354822491878604}

In [ ]:
french_artist_dict.keys()

dict_keys(['anpil', 'antoine', 'elodie', 'ekipman', 'kk', 'vwa', 'depi', 'aptman', 'b', 'jeanpierre', 'tann', 'playlist', 'kase', 'lonbraj', 'd', 'tab', 'anj', 'melanje', 'lanp', 'estidyo'])

Verb

In [ ]:
df_artist_verb_haitian = pd.read_csv('Haitian_Artists_Verb_Story.csv', header=None)
df_artist_verb_french = pd.read_csv('French_Artists_Verb_Story.csv',header=None)
df_artist_verb_all = pd.concat([df_artist_verb_haitian, df_artist_verb_french], ignore_index=True)

In [ ]:
artits_verb_results = get_log_odds(get_text_column(df_artist_verb_haitian),get_text_column(df_artist_verb_french),get_text_column(df_artist_verb_all),verbose=False,lower=True)

In [ ]:
haitian_artist_verb = {}
french_artist_verb = {}
for word , score  in artits_verb_results.items():
  if score > 1.2 and word not in STOPWORDS_HT:
    haitian_artist_verb[word] = artits_verb_results[word]
  if score <- 1.2 and word not in STOPWORDS_HT:
    french_artist_verb[word] = artits_verb_results[word]

In [ ]:
haitian_artist_verb = dict(sorted(haitian_artist_verb.items(), key=lambda item: item[1], reverse=True))
haitian_artist_verb

{'tande': 2.44584583697037,
 'tanbou': 2.15646400602371,
 'papa': 1.861530516239659,
 'avk': 1.7912720487844767,
 'j': 1.7770889311497464,
 'tap': 1.6281744489171244,
 'gita': 1.6085840132073193,
 'perry': 1.607319263746096,
 'chof': 1.5153394419155821,
 'bat': 1.5040463820797199,
 'zami': 1.4442925936682705,
 'peye': 1.4164043065534773,
 'nt': 1.3446826740649502,
 'pouki': 1.3122260005637223,
 'nwaj': 1.3122260005637223,
 'sajs': 1.3122260005637223,
 'lakou': 1.3075007111300665,
 'wout': 1.3075007111300665,
 'priy': 1.3075007111300665,
 'pr': 1.2386004503832055,
 'do': 1.2183924776078738}

In [ ]:
french_artist_verb = dict(sorted(french_artist_verb.items(), key=lambda item: item[1]))
french_artist_verb

{'km': -6.278551100722322,
 'kriye': -4.789034163764687,
 'rive': -3.590152558529312,
 'tannare': -2.7347486786534256,
 'sonje': -2.542979265046407,
 'le': -1.9785351668347997,
 'tann': -1.9680667709831032,
 'sonte': -1.8863896312644004,
 'je': -1.7916826018614036,
 'finis': -1.5780920514069152,
 'v': -1.5780920514069152,
 'fmen': -1.5113564749748303,
 'frans': -1.4609754205340089,
 'vinil': -1.4609754205340089,
 'lag': -1.4609754205340089,
 'kafe': -1.426166442225914,
 'gade': -1.4226458847381223,
 'bar': -1.3851609742044222,
 'tounen': -1.3394334974326139,
 'mots': -1.3336327714835374,
 'un': -1.3336327714835374,
 'dos': -1.3336327714835374,
 'marie': -1.3336327714835374,
 'desn': -1.3336327714835374,
 'zy': -1.3336327714835374,
 'ble': -1.3266009886757235,
 'pas': -1.3000969682828485,
 'pe': -1.2770487278683864,
 'sti': -1.2547528315729857,
 'achte': -1.217442659694338}

Names story

In [ ]:
df_haitian_person = pd.read_csv('Haitian_Name_adjective_Story.csv', header=None)
df_french_person = pd.read_csv('French_Name_adjective_Story.csv',header=None)
df_all_person = pd.concat([df_haitian_person, df_french_person], ignore_index=True)

In [ ]:
df_haitian_person

,0
0,Nan yon ti vilaj kote solèy la toujou parèt ak...
1,Nan yon ti vilaj kote solèy la toujou klere ak...
2,Nan yon ti vilaj kote solèy la toujou parèt ak...
3,Nan yon ti vilaj kote solèy la toujou klere ak...
4,Nan yon kote kote solèy la koule dousman sou t...
...,...
146,Nan yon ti vilaj kote solèy la toujou parèt ak...
147,Solèy la te kòmanse kouche sou lèt kote li te ...
148,Jean-Baptiste te grandi nan yon ti vil nan Ayi...
149,"Nan yon ti vil kote solèy la toujou klere, Tam..."


In [ ]:
person_results = get_log_odds(get_text_column(df_haitian_person),get_text_column(df_french_person),get_text_column(df_all_person),verbose=False,lower=True)

In [ ]:
haitian_person_dict = defaultdict(float)
french_person_dict = defaultdict(float)
for word , score  in person_results.items():
  if score > 1.9 and word not in STOPWORDS_HT:
    haitian_person_dict[word] = person_results[word]
  if score < -1.9  and word not in STOPWORDS_HT:
    french_person_dict[word] = person_results[word]

In [ ]:
haitian_person_dict = dict(sorted(haitian_person_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_person_dict

{'ayisyen': 5.962486403834434,
 'gason': 5.16236642933343,
 'grandi': 4.958094972025218,
 'toujou': 4.8254889618985075,
 'rele': 4.562881597786776,
 'lavi': 4.475664881970096,
 'klere': 4.393185585719898,
 'teye': 4.23540464416027,
 'chche': 4.177155655246633,
 'ayiti': 3.652092616737483,
 'viv': 3.5962632120248594,
 'vilaj': 3.5368957728200163,
 'pote': 3.421230881666695,
 'kilti': 3.336128262991047,
 'lespri': 3.3021517574624824,
 'val': 3.0994984581776075,
 'van': 2.7120511443975466,
 'konnen': 2.4894032249925853,
 'fs': 2.4885427035193044,
 'part': 2.467863390836623,
 'travay': 2.42822739079106,
 'pase': 2.41828319416602,
 'souri': 2.113387954768451,
 'rankontre': 2.0815924748323065,
 'km': 2.0734000103188484,
 'istwa': 2.0504255360355086,
 'aprann': 2.040132262571303,
 'je': 1.9985748884553955,
 'laj': 1.951597591768373,
 'chal': 1.9201144177997993,
 'fon': 1.903268582498708,
 'rivy': 1.90112830276671}

In [ ]:
haitian_person_dict.keys()

dict_keys(['ayisyen', 'gason', 'grandi', 'toujou', 'rele', 'lavi', 'klere', 'teye', 'chche', 'ayiti', 'viv', 'vilaj', 'pote', 'kilti', 'lespri', 'val', 'van', 'konnen', 'fs', 'part', 'travay', 'pase', 'souri', 'rankontre', 'km', 'istwa', 'aprann', 'je', 'laj', 'chal', 'fon', 'rivy'])

In [ ]:
french_person_dict = dict(sorted(french_person_dict.items(), key=lambda item: item[1]))

In [ ]:
french_person_dict

{'leve': -7.114490232534618,
 'ti': -6.925408392996818,
 'jounen': -6.376373679154371,
 'kmanse': -6.040187982278766,
 'kafe': -5.335971831654311,
 'pran': -4.535826549573426,
 'souf': -4.1295223948174105,
 'kl': -3.9441296134355825,
 'frans': -3.8696317841102954,
 'kras': -3.773656114292177,
 'paris': -3.6016129668855577,
 'maten': -3.4475202145332164,
 'ouvri': -3.261554721793486,
 'kwizin': -3.261554721793486,
 'santi': -3.2428674865728406,
 'prepare': -3.0761394166651024,
 'lt': -3.0315952851524757,
 'mete': -2.991457047612639,
 'apatman': -2.9392583445718814,
 'limy': -2.9026521631068727,
 'fent': -2.823734417222987,
 'kouvri': -2.6619104404335987,
 'gou': -2.5773150719108036,
 'kalm': -2.5746677725645952,
 'bri': -2.5510038160433215,
 'monte': -2.5372273932654816,
 'kase': -2.444870818081863,
 'santiman': -2.3942590124022747,
 'tas': -2.304871972946578,
 'sant': -2.187955600405947,
 'kofi': -2.155847256479469,
 'ritm': -2.059837417202867,
 'kaf': -2.0321400447059057,
 'fatig': -1

In [ ]:
french_person_dict.keys()

dict_keys(['leve', 'ti', 'jounen', 'kmanse', 'kafe', 'pran', 'souf', 'kl', 'frans', 'kras', 'paris', 'maten', 'ouvri', 'kwizin', 'santi', 'prepare', 'lt', 'mete', 'apatman', 'limy', 'fent', 'kouvri', 'gou', 'kalm', 'bri', 'monte', 'kase', 'santiman', 'tas', 'sant', 'kofi', 'ritm', 'kaf', 'fatig', 'dous', 'kouri', 'pwomt'])

Verb

In [ ]:
df_haitian_person_verb = pd.read_csv('Haitian_Name_Verb_Story.csv', header=None)
df_french_person_verb = pd.read_csv('French_Name_Verb_Story.csv',header=None)
df_all_person_verb = pd.concat([df_haitian_person_verb, df_french_person_verb], ignore_index=True)

In [ ]:
person_verb_results = get_log_odds(get_text_column(df_haitian_person_verb),get_text_column(df_french_person_verb),get_text_column(df_all_person_verb),verbose=False, lower=True)

In [ ]:
haitian_person_verb = defaultdict(float)
french_person_verb = defaultdict(float)
for word , score  in person_verb_results .items():
  if score > 1.2 and word not in STOPWORDS_HT:
    haitian_person_verb[word] = person_verb_results [word]
  if score < -1.2  and word not in STOPWORDS_HT:
    french_person_verb[word] = person_verb_results [word]

In [ ]:
haitian_person_verb = dict(sorted(haitian_person_verb.items(), key=lambda item: item[1], reverse=True))
haitian_person_verb

{'klara': 2.712977439271839,
 'osvaldo': 2.5865211538540276,
 'kote': 1.7732855055284111,
 'renmen': 1.7558129196981993,
 'jwenn': 1.733627614217881,
 'f': 1.7266489326284706,
 'kl': 1.6529647541891566,
 'repnn': 1.5876122563476185,
 'gade': 1.530451621061509,
 'du': 1.4159245326901733,
 'tann': 1.3959112542405763,
 'avk': 1.3687215329233429,
 'souriait': 1.3687215329233429,
 'konnen': 1.3659332650589722,
 'koute': 1.3259827393764558,
 'bel': 1.2925060719952712,
 'chire': 1.2925060719952712,
 'prepare': 1.2925060719952712,
 'dernire': 1.2925060719952712,
 'vivan': 1.2925060719952712,
 'antiquaire': 1.2925060719952712,
 'al': 1.2925060719952712,
 'kouleur': 1.2925060719952712,
 'cadre': 1.2925060719952712,
 'pran': 1.2593753802288108,
 'ft': 1.246669244508271,
 'pn': 1.246669244508271,
 'traner': 1.246669244508271,
 'valse': 1.246669244508271,
 'menm': 1.246669244508271,
 'vire': 1.246669244508271,
 'ayisyen': 1.246669244508271,
 'lt': 1.2438756857709479,
 'vle': 1.237425879167905,
 'ga

In [ ]:
french_person_verb = dict(sorted(french_person_verb.items(), key=lambda item: item[1]))
french_person_verb

{'regarda': -1.7525527696398724,
 'monte': -1.6790539498148465,
 'moi': -1.6625529746033665,
 'pain': -1.602809741781277,
 'enfum': -1.567409107210274,
 'rivy': -1.484221796084244,
 'garon': -1.466120032545045,
 'lcher': -1.466120032545045,
 'rparer': -1.466120032545045,
 'prendre': -1.4044573616209626,
 'jus': -1.4044573616209626,
 'ai': -1.4044573616209626,
 'librer': -1.3573106889009559,
 'faim': -1.3573106889009559,
 'fruit': -1.3573106889009559,
 'donne': -1.3573106889009559,
 'pleurer': -1.3573106889009559,
 'goter': -1.3573106889009559,
 'travailler': -1.3573106889009559,
 'maintenant': -1.3573106889009559,
 'cest': -1.3573106889009559,
 'apprendre': -1.2951744531753564,
 'wi': -1.2951744531753564,
 'sentir': -1.2798100156394259,
 'hocha': -1.239001359943581,
 'crier': -1.239001359943581,
 'sche': -1.239001359943581,
 'gagner': -1.239001359943581,
 'pardonner': -1.239001359943581,
 'retourner': -1.239001359943581,
 'chemise': -1.239001359943581,
 'poids': -1.239001359943581,
 'f